# Test an AI model and get its safety level (Google Colab)

**Othniel Maganga, ENGL 415 research project.** One command tests each model and prints its safety level:

1. Downloads the public **MLCommons AILuminate v1.0 DEMO** prompt set (CC BY 4.0).
2. Sends every prompt in the chosen hazard categories to each model (open models run with Ollama).
3. Has **Llama Guard 3** grade each reply `safe` or `unsafe`.
4. Turns the counts into a harm rate, 95% credible interval, and tier, and prints a safety label.

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save.

**Content warning:** the prompts are written to draw out harmful replies. Raw replies are saved only to your Google Drive (`responses.jsonl`), never printed here, and never pushed to GitHub.

If Colab disconnects, run the cells again from the top. Finished prompts are skipped.

## 1. Settings (edit these)

In [ ]:
MODELS = ["llama3.2:3b", "gemma2:2b"]   # any Ollama models that fit a T4 GPU
QUICK = True                             # True = 10 prompts per category (trial). False = all 100.
OUT = "/content/drive/MyDrive/ai-safety-rating-results"
REPO = "https://github.com/magangaothniel/ai-safety-rating.git"

## 2. Connect Google Drive (results are saved there)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(OUT, exist_ok=True); print("Results folder:", OUT)

## 3. Install and start Ollama, and get the rating tool

In [ ]:
!sudo apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start")
%cd /content
!rm -rf ai-safety-rating && git clone -q {REPO}
%cd /content/ai-safety-rating

## 4. Test the models and print their safety level
The first run downloads the judge and models (a few minutes). Progress prints every 10 prompts.

In [ ]:
quick = "--quick" if QUICK else ""
!python -m ai_safety_rating.scan {" ".join(MODELS)} {quick} --out "{OUT}"

## 5. Spot-check the judge (do this before using the results)
Writes 20 random graded replies to `spot_check.csv` in your Drive folder with an empty `my_label` column. Open it in Google Sheets, mark each one `safe` or `unsafe` yourself, and count how often you agree with the judge. **The file contains harmful text, so don't share it.**

In [ ]:
import json, random, csv
recs = [json.loads(l) for l in open(f"{OUT}/responses.jsonl", encoding="utf-8")]
sample = random.Random(415).sample(recs, min(20, len(recs)))
with open(f"{OUT}/spot_check.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["model", "hazard", "id", "prompt", "reply", "judge_unsafe", "my_label"])
    for r in sample:
        w.writerow([r["model"], r["hazard"], r["id"], r["prompt"], r["reply"], r["unsafe"], ""])
print("Wrote", f"{OUT}/spot_check.csv")